In [0]:
display(
    dbutils.fs.ls(
        "s3://sales-streaming-medallion-pipeline-jhansi/incoming/sales/"
    )
)

In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

In [0]:
# Source and metadata paths
source_path = "s3://sales-streaming-medallion-pipeline-jhansi/incoming/sales/"

schema_location = "s3://sales-streaming-medallion-pipeline-jhansi/databricks_metadata/sales_schema/"

checkpoint_location = "s3://sales-streaming-medallion-pipeline-jhansi/databricks_metadata/sales_checkpoint/"

target_table = "sales_streaming.sales.sales_bronze"

In [0]:
sales_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("rescuedDataColumn", "_rescued_data")
        .load(source_path)
)

In [0]:
sales_stream.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

sales_bronze = (
    sales_stream
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", input_file_name())
)

bronze_query = (
    sales_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_location)
    .trigger(availableNow=True)
    .toTable(target_table)
)